# 01 - BB84 with Qiskit: the quantum core of QuantumSentinel

Q-Hack India 2026 | Team Quantum Syndicate | Quantum Security & Cryptography track

**What this notebook shows:** BB84 quantum key distribution built as *real quantum circuits* on the IBM Qiskit Aer simulator (stabilizer method), including an intercept-resend eavesdropper (Eve) and Pauli noise channels. This is the exact logic the FastAPI backend executes per session.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, pauli_error, depolarizing_error

sim = AerSimulator(method='stabilizer')   # handles thousands of independent qubits

def bb84_circuit(bits, a_bases, b_bases, eve_flip):
    """One qubit per BB84 signal. Eve's intercept-resend = stochastic X flip."""
    k = len(bits)
    qc = QuantumCircuit(k, k)
    for i in range(k):
        if bits[i]:    qc.x(i)   # Alice: bit value
        if a_bases[i]: qc.h(i)   # Alice: encode in X basis
        qc.id(i)                 # channel (noise attaches here)
        if eve_flip[i]: qc.x(i)  # Eve intercept-resend flip
        if b_bases[i]: qc.h(i)   # Bob: decode
        qc.measure(i, i)
    return transpile(qc, sim)


In [ ]:
rng = np.random.default_rng(1)
n = 2000
a, ba, bb = rng.integers(0, 2, n), rng.integers(0, 2, n), rng.integers(0, 2, n)
f = 0.5                                   # Eve strength 50%
eve_flip = (rng.random(n) < f) & (rng.random(n) < 0.5)

counts = sim.run(bb84_circuit(a, ba, bb, eve_flip), shots=1).result().get_counts()
bitstr = next(iter(counts))               # 1 shot -> one outcome per qubit
bob = np.array([int(bitstr[n-1-i]) for i in range(n)])

sift = ba == bb
er = (a[sift] != bob[sift])
print(f'sifted key length: {sift.sum()} bits')
print(f'QBER with Eve    : {er.mean():.3f}  <- far above the 11% abort threshold')

eve_flip[:] = False                       # same channel, Eve off
counts = sim.run(bb84_circuit(a, ba, bb, eve_flip), shots=1).result().get_counts()
bitstr = next(iter(counts)); bob = np.array([int(bitstr[n-1-i]) for i in range(n)])
er = (a[sift] != bob[sift])
print(f'QBER without Eve : {er.mean():.3f}  <- key would be kept')


In [ ]:
# Depolarizing noise on the channel, visualised on a small circuit
nm = NoiseModel()
nm.add_all_qubit_quantum_error(depolarizing_error(0.1, 1), ['id'])
small = bb84_circuit(a[:6], ba[:6], bb[:6], eve_flip[:6])
print(small.draw())
counts = sim.run(small, noise_model=nm, shots=1024).result().get_counts()
print('noisy sample counts:', dict(list(counts.items())[:4]), '...')


**Takeaway:** every session in the demo UI is produced by circuits exactly like these - real gates, real measurements, real noise models on Qiskit Aer. The JS frontend keeps a statistically identical Monte-Carlo twin purely as a classical baseline for comparison.